# HIT140 Assessment 2: Defender age and fouls at the FIFA World Cup 2026

**Analytic question:** Among defenders who played at the FIFA World Cup 2026, do older defenders commit fewer fouls per 90 minutes than younger defenders? Only defenders with at least N minutes on the field are included.

## Data Wrangling

### Data Acquisition

Player statistics are taken from FBref. Two tables for the 2026 World Cup were exported using the CSV export feature of the site.

| File | Table Name | Source |
|---|---|---|
| `data/raw/fbref_player_standard.csv` | Player Standard Stats (position, age, minutes, cards) | https://fbref.com/en/comps/1/stats/World-Cup-Stats |
| `data/raw/fbref_player_misc.csv` | Player Miscellaneous Stats (fouls, cards, tackles) | https://fbref.com/en/comps/1/misc/World-Cup-Stats |

### Cleaning

Opening the file in a text editor shows the table starts with the labels on the 6th row, so we skip the first 5 rows to get the data only

```
--- When using SR data, please cite us and provide a link and/or a mention.


 
,,,,,,,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,Performance,,-additional
Rk,Player,Pos,Squad,Age,Born,90s,CrdY,CrdR,2CrdY,Fls,Fld,Off,Crs,Int,TklW,PKwon,PKcon,OG,Matches,-9999
1,Brenden Aaronson,MF,us USA,25,2000,0.8,0,0,0,2,1,0,1,0,1,,,0,Matches,5bc43860
2,Thelo Aasgaard,MF,no Norway,24,2002,1.0,0,0,0,0,3,0,0,3,1,,,0,Matches,c88a28b9
...
```

In [1]:
import pandas as pd

standard = pd.read_csv("data/raw/fbref_player_standard.csv", skiprows=5)
misc = pd.read_csv("data/raw/fbref_player_misc.csv", skiprows=5)

print("standard:", standard.shape)
print("misc:", misc.shape)

standard.head()

standard: (1040, 26)
misc: (1039, 21)

,Rk,Player,Pos,Squad,Age,Club,Born,MP,Starts,Min,...,PKatt,CrdY,CrdR,Gls.1,Ast.1,G+A.1,G-PK.1,G+A-PK,Matches,-9999
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-9999
1,1.0,Brenden Aaronson,MF,us USA,25.0,1.eng Leeds United,2000.0,1.0,1.0,76.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,5bc43860
2,2.0,Thelo Aasgaard,MF,no Norway,24.0,1.sct Rangers,2002.0,1.0,1.0,90.0,...,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,Matches,c88a28b9
3,3.0,Hamza Abdelkarim,FW,eg Egypt,18.0,4.es Barcelona B,2008.0,4.0,0.0,60.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,057d8f12
4,4.0,Hossam Abdelmaguid,DF,eg Egypt,25.0,1.eg Zamalek SC,2001.0,2.0,0.0,59.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Matches,accbfc94


In [2]:
# Check which columns we have
print("columns in standard:", list(standard.columns))
print()
print("columns in misc:", list(misc.columns))

columns in standard: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Club', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK', 'Matches', '-9999']

columns in misc: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', '90s', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Fld', 'Off', 'Crs', 'Int', 'TklW', 'PKwon', 'PKcon', 'OG', 'Matches', '-9999']

Columns kept for this question (names as FBref abbreviates them, with FBref's glossary meaning where it isn't obvious):

| Table | Column | Meaning |
|---|---|---|
| both | `player_id` | FBref's player identifier (originally labelled -9999) for join |
| standard | `Player` | Player name |
| standard | `Pos` | Position: GK, DF, MF, FW, two positions per player based on time in role |
| standard | `Squad` | National team |
| standard | `Age` | Age in years at the time of export |
| standard | `MP` | Matches played |
| standard | `Starts` | Matches started |
| standard | `Min` | Minutes played |
| standard | `90s` | Minutes played divided by 90 (number of full matches' worth of playing time) |
| misc | `Fls` | Fouls committed |
| misc | `Fld` | Fouls drawn (fouls suffered) |
| misc | `CrdY` | Yellow cards |
| misc | `CrdR` | Red cards |
| misc | `2CrdY` | Second yellow card in a match (counted as a red) |

Everything else is dropped: the row number `Rk`, the `Matches` link column, club, goals and assists, the per-90 goal columns, offsides, crosses, interceptions, tackles won, own goals, and the penalty columns (`PKwon`, `PKcon`), which are empty for every player in this export.

In [3]:
# Remove the NaN junk row
standard = standard.dropna(subset=["Player"])
# And reinfer types after dropping junk rows
standard = standard.convert_dtypes()

# Label the player id row, which is labelled -9999 in the raw data
standard = standard.rename(columns={"-9999": "player_id"})
misc = misc.rename(columns={"-9999": "player_id"})

# select columns useful for this question
standard = standard[["player_id", "Player", "Pos", "Squad", "Age", "MP", "Starts", "Min", "90s"]]
misc = misc[["player_id", "Fls", "Fld", "CrdY", "CrdR", "2CrdY"]]

# remove the country code from Squad
standard["Squad"] = standard["Squad"].str.split(" ", n=1).str[1]

print("standard:", standard.shape)
print("misc:", misc.shape)
standard.head()

standard: (1039, 9)
misc: (1039, 6)

,player_id,Player,Pos,Squad,Age,MP,Starts,Min,90s
1,5bc43860,Brenden Aaronson,MF,USA,25,1,1,76,0.8
2,c88a28b9,Thelo Aasgaard,MF,Norway,24,1,1,90,1.0
3,057d8f12,Hamza Abdelkarim,FW,Egypt,18,4,0,60,0.7
4,accbfc94,Hossam Abdelmaguid,DF,Egypt,25,2,0,59,0.7
5,bff15d74,Mohamed Abdelmonem,DF,Egypt,27,2,1,14,0.2
